# HW5: Transition Matrix from Errored Traces

In this homework you'll upload synthetic traces with tool call errors to LangSmith, then pull them down and build a failure transition matrix.

**What you'll learn:**
- How to upload pre-built traces to LangSmith
- How to use LangSmith dashboards to view error patterns
- How to pull traces via the SDK and analyze them programmatically
- How to build and visualize a transition heatmap

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 1. Upload Traces to LangSmith

The file `traces.json` contains pre-generated synthetic traces for a recipe agent.
Some traces include tool call errors (timeouts, 503s, connection resets).

We use `upload_traces.py` to push them to LangSmith with fresh timestamps.

In [ ]:
from upload_traces import upload

PROJECT_NAME = "recipe-agent-traces"

upload(project=PROJECT_NAME)

## 2. View the Dashboard (UI)

Open [smith.langchain.com](https://smith.langchain.com) and navigate to the **recipe-agent-traces** project.

### What to look for:
- **Monitor tab:** Check the error rate over time
- **Traces tab:** Filter by `error = true` to see which traces have failed tool calls
- Click into individual traces to see the full execution tree — you'll see the failed tool call, retry, and eventual response

Notice how some traces have a pattern: `LLM → web_search (error) → LLM (retry) → web_search (success) → LLM (response)`

## 3. Pull Traces via SDK

Let's pull down all traces from the project and analyze the error patterns programmatically.

In [ ]:
from langsmith import Client

client = Client()

# Get all runs from the project
runs = list(client.list_runs(project_name=PROJECT_NAME))
print(f"Total runs: {len(runs)}")

# Separate root runs (traces) from child runs
root_runs = [r for r in runs if r.parent_run_id is None]
child_runs = [r for r in runs if r.parent_run_id is not None]

print(f"Traces (root runs): {len(root_runs)}")
print(f"Child runs: {len(child_runs)}")

In [ ]:
# Group child runs by trace
from collections import defaultdict

trace_children = defaultdict(list)
for run in child_runs:
    trace_children[run.trace_id].append(run)

# Sort children by start_time within each trace
for trace_id in trace_children:
    trace_children[trace_id].sort(key=lambda r: r.start_time)

## 4. Analyze Error Patterns

For each trace, extract the sequence of steps and identify where errors occur.

In [ ]:
# Build step sequences for each trace
trace_sequences = []

for root in root_runs:
    children = trace_children.get(root.trace_id, [])
    steps = []
    for child in children:
        step_name = child.name
        if child.run_type == "tool":
            step_name = f"tool:{child.name}"
        elif child.run_type == "llm":
            step_name = "llm"

        status = "error" if child.error else "ok"
        steps.append((step_name, status))

    trace_sequences.append({
        "trace_id": str(root.trace_id),
        "steps": steps,
        "has_error": any(s[1] == "error" for s in steps),
    })

error_traces = [t for t in trace_sequences if t["has_error"]]
print(f"Traces with errors: {len(error_traces)} / {len(trace_sequences)}")

In [ ]:
# Show a few example sequences
print("Example error traces:")
for t in error_traces[:5]:
    seq = " → ".join(f"{name}({'!' if status == 'error' else '✓'})" for name, status in t["steps"])
    print(f"  {seq}")

## 5. Build the Transition Matrix

For each trace, find the **last successful step** before an error occurs. Build a matrix of `(last_ok_step → error_step)` transitions to see which step transitions are most likely to fail.

In [ ]:
import pandas as pd

transitions = []

for trace in trace_sequences:
    steps = trace["steps"]
    for i, (name, status) in enumerate(steps):
        if status == "error":
            # Find the last successful step before this error
            prev_ok = "start"
            for j in range(i - 1, -1, -1):
                if steps[j][1] == "ok":
                    prev_ok = steps[j][0]
                    break
            transitions.append({"from_step": prev_ok, "to_step": name})

trans_df = pd.DataFrame(transitions)
print(f"Total error transitions: {len(trans_df)}")
trans_df.head(10)

In [ ]:
# Build the transition matrix
matrix = trans_df.groupby(["from_step", "to_step"]).size().unstack(fill_value=0)
print("Transition matrix (last OK step → errored step):")
matrix

## 6. Visualize as Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
)

ax.set_title("Error Transition Matrix\n(Last Successful Step → Errored Step)", fontsize=13)
ax.set_xlabel("Errored Step", fontsize=11)
ax.set_ylabel("Last Successful Step", fontsize=11)

plt.tight_layout()
plt.show()

## 7. Interpretation

The heatmap shows which step transitions are most prone to errors.

**What to look for:**
- Which tool calls fail most often after an LLM call?
- Are there patterns in _when_ errors occur in the execution sequence?
- How does the retry pattern (error → retry → success) show up in the data?

This kind of analysis helps you:
1. Identify unreliable tool integrations
2. Decide where to add retry logic or fallbacks
3. Prioritize which error patterns to fix first